# NB04b — Context-repaired full rerun
## Prompt-only vs Frozen Static RAG · OpenAI GPT-6 Luna

This notebook reruns **all Q1–Q10** after fixing a benchmark-execution issue found in the initial NB04 run.

Some benchmark questions are context-dependent:
- Q5: “If so”
- Q8: “the assessment”
- Q9: “each major conclusion”
- Q10: “the evidence above”

The original NB04 sent every question independently. This repaired version adds one fixed shared context to **every question in both modes**.

### What changes
`BENCHMARK_CONTEXT + current question`

### What does not change
- same Q1–Q10 benchmark
- same system prompt
- same model / reasoning effort
- same frozen Build02 Top-5 retrieval
- no query rewrite
- no retrieval rerun
- no gold/reference answers in generation
- no previous model answers in generation

## Experimental interpretation

Keep the original NB04 output as:

**Initial run — context dependency discovered**

Use this notebook as the cleaner official Build01 vs Build02 generation comparison.

In [1]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

from pathlib import Path
import json, hashlib, os
import pandas as pd
from datetime import datetime, timezone

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"

BENCHMARK_PATH = __import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json")
CHUNKS_PATH = __import__("pathlib").Path("data/processed/chunks_legal_aware.json")
RETRIEVAL_CSV_PATH = __import__("pathlib").Path("data/processed/retrieval_results_nb03_final_hybrid_static_rag.csv")
RETRIEVAL_CONFIG_PATH = __import__("pathlib").Path("data/processed/retrieval_config_nb03_final_hybrid_static_rag.json")

for p in [BENCHMARK_PATH, CHUNKS_PATH, RETRIEVAL_CSV_PATH, RETRIEVAL_CONFIG_PATH]:
    print(f"{'✓' if p.exists() else '✗'}  {p}")

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


✓  data\benchmark_questions_v1.json
✓  data\chunks_legal_aware.json
✓  data\retrieval_results_nb03_final_hybrid_static_rag.csv
✓  data\retrieval_config_nb03_final_hybrid_static_rag.json


In [2]:
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

with open(RETRIEVAL_CONFIG_PATH, "r", encoding="utf-8") as f:
    retrieval_config = json.load(f)

retrieval_df = pd.read_csv(RETRIEVAL_CSV_PATH)

# Normalize benchmark_questions_v1.json into:
# [{"qid": "Q1", "question": "..."}, ...]
#
# Expected current format:
# {
#   "benchmark_id": "...",
#   "source": "...",
#   "questions": {
#       "Q1": "...",
#       ...
#   }
# }

if isinstance(benchmark, dict) and "questions" in benchmark:
    raw_questions = benchmark["questions"]

    if isinstance(raw_questions, dict):
        questions = [
            {"qid": str(qid), "question": str(question)}
            for qid, question in raw_questions.items()
        ]
    elif isinstance(raw_questions, list):
        questions = [
            {"qid": str(x["qid"]), "question": str(x["question"])}
            for x in raw_questions
        ]
    else:
        raise TypeError(
            f"Unexpected benchmark['questions'] type: {type(raw_questions).__name__}"
        )

elif isinstance(benchmark, list):
    questions = [
        {"qid": str(x["qid"]), "question": str(x["question"])}
        for x in benchmark
    ]

else:
    raise TypeError(
        "Unexpected benchmark format. Expected a dict containing 'questions' "
        "or a list of {'qid', 'question'} records."
    )

# Keep Q1 ... Q10 in numeric order.
questions = sorted(
    questions,
    key=lambda x: int(x["qid"].lstrip("Q"))
)

qid_to_question = {x["qid"]: x["question"] for x in questions}

print("Benchmark ID:", benchmark.get("benchmark_id") if isinstance(benchmark, dict) else None)
print("Benchmark questions:", len(questions))
print("Chunks:", len(chunks) if hasattr(chunks, "__len__") else "?")
print("Retrieval rows:", len(retrieval_df))

for x in questions:
    print(x["qid"], "—", x["question"])

Benchmark ID: regulatory_agent_v1_notion_2026-10-08
Benchmark questions: 10
Chunks: 1383
Retrieval rows: 50
Q1 — A shipment of dried figs originating from Türkiye is found to contain aflatoxin B1. Which EU legislation is relevant to this case?
Q2 — What is the applicable EU maximum level for aflatoxin B1 and total aflatoxins in dried figs intended for direct human consumption?
Q3 — Does the applicable limit change depending on whether the dried figs are intended for direct consumption or further processing/sorting?
Q4 — Are dried figs from Türkiye subject to increased official controls or special import conditions when entering the EU?
Q5 — If so, what controls, sampling/testing requirements, or documentation requirements currently apply?
Q6 — Which authorities or food business operators are responsible for the relevant actions when a non-compliant consignment is detected?
Q7 — What regulatory actions may follow if aflatoxin levels exceed the applicable maximum level?
Q8 — Which parts 

## Fixed shared benchmark context

This exact block is supplied to **all 20 generations**:
- 10 Prompt-only
- 10 RAG

It restores only the shared-case semantics. It does not provide any legal answer.

In [3]:
BENCHMARK_CONTEXT = """
All questions belong to one fixed EU food-regulation benchmark concerning the same case:
a shipment of dried figs originating from Türkiye in which aflatoxin B1 has been detected.

Q1–Q10 form one regulatory-assessment sequence. Answer only the current question, but when
the wording refers to “if so”, “the assessment”, “major conclusions”, “the evidence above”,
or similar cross-question references, interpret those references within this shared benchmark case.

Across the sequence, the regulatory issues being examined include:
relevant EU legislation, contaminant limits, product-use/processing distinctions,
import controls and documentation, responsible actors, possible regulatory actions,
and which conclusions require current-law verification.

Do not assume shipment-specific facts that are not stated in the current question or supplied evidence.
Do not invent legal provisions, numerical limits, or documentary requirements.
""".strip()

context_sha256 = hashlib.sha256(BENCHMARK_CONTEXT.encode("utf-8")).hexdigest()

print(BENCHMARK_CONTEXT)
print("\nSHA256:", context_sha256)

All questions belong to one fixed EU food-regulation benchmark concerning the same case:
a shipment of dried figs originating from Türkiye in which aflatoxin B1 has been detected.

Q1–Q10 form one regulatory-assessment sequence. Answer only the current question, but when
the wording refers to “if so”, “the assessment”, “major conclusions”, “the evidence above”,
or similar cross-question references, interpret those references within this shared benchmark case.

Across the sequence, the regulatory issues being examined include:
relevant EU legislation, contaminant limits, product-use/processing distinctions,
import controls and documentation, responsible actors, possible regulatory actions,
and which conclusions require current-law verification.

Do not assume shipment-specific facts that are not stated in the current question or supplied evidence.
Do not invent legal provisions, numerical limits, or documentary requirements.

SHA256: 6d540ebf6b35f579ff490e3f94becc04634a361a6fec8bfedbb6b

### Why this is the cleanest repair

The context:
- gives no regulation numbers,
- gives no article/annex numbers,
- gives no legal thresholds,
- gives no gold answer,
- gives no previous model answers.

It only clarifies what cross-question references point to.

Because it is supplied identically to every question and both architectures, the experiment remains fair.

In [4]:
expected = {
    "build": "Build02",
    "retrieval_type": "frozen_static_rag",
    "query_mode": "single_query_single_shot",
    "retrieval_method": "hybrid_dense_bm25_rrf_diverse_v1",
    "top_k": 5,
}

for k, v in expected.items():
    actual = retrieval_config.get(k)
    assert actual == v, f"{k}: expected {v!r}, got {actual!r}"

assert retrieval_config.get("query_decomposition") is False
assert retrieval_config.get("llm_used_in_retrieval") is False
assert retrieval_config.get("iterative_retrieval") is False

print("✓ Frozen Build02 retrieval config verified")

✓ Frozen Build02 retrieval config verified


In [5]:
print(retrieval_df.columns.tolist())
display(retrieval_df.head())

['qid', 'question', 'benchmark_id', 'benchmark_sha256', 'strategy', 'retrieval_method', 'rank', 'score', 'dense_cosine_score', 'bm25_score', 'chunk_id', 'file', 'version_date', 'location', 'article', 'heading', 'text']


,qid,question,benchmark_id,benchmark_sha256,strategy,retrieval_method,rank,score,dense_cosine_score,bm25_score,chunk_id,file,version_date,location,article,heading,text
0,Q1,A shipment of dried figs originating from Türk...,regulatory_agent_v1_notion_2026-10-08,2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a...,legal_aware,hybrid_dense_bm25_rrf_diverse_v1,1,0.032787,0.572145,31.089920,legal::EU_2019_1793_consolidated_2026-06-30.ht...,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,NaN,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,Q1,A shipment of dried figs originating from Türk...,regulatory_agent_v1_notion_2026-10-08,2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a...,legal_aware,hybrid_dense_bm25_rrf_diverse_v1,2,0.028787,0.534844,28.902159,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 3, row 6",NaN,NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
2,Q1,A shipment of dried figs originating from Türk...,regulatory_agent_v1_notion_2026-10-08,2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a...,legal_aware,hybrid_dense_bm25_rrf_diverse_v1,3,0.027047,0.535315,20.485682,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 3, row 14",NaN,NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
3,Q1,A shipment of dried figs originating from Türk...,regulatory_agent_v1_notion_2026-10-08,2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a...,legal_aware,hybrid_dense_bm25_rrf_diverse_v1,4,0.026743,0.535330,20.347683,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 3, row 12",NaN,NaN,COMMISSION REGULATION (EU) 2023/915. Table con...
4,Q1,A shipment of dried figs originating from Türk...,regulatory_agent_v1_notion_2026-10-08,2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a...,legal_aware,hybrid_dense_bm25_rrf_diverse_v1,5,0.025978,0.534914,20.363025,legal::EU_2023_915_consolidated_2026-08-19.htm...,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 3, row 10",NaN,NaN,COMMISSION REGULATION (EU) 2023/915. Table con...


In [6]:
def resolve_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

QID_COL = resolve_col(retrieval_df, ["qid", "QID"])
QUESTION_COL = resolve_col(retrieval_df, ["question", "query"])
RANK_COL = resolve_col(retrieval_df, ["rank", "retrieval_rank", "final_rank"])
CHUNK_ID_COL = resolve_col(retrieval_df, ["chunk_id", "id"])
TEXT_COL = resolve_col(retrieval_df, ["text", "chunk_text", "content"])
SOURCE_COL = resolve_col(retrieval_df, ["source", "source_file", "filename", "file"])
LOCATION_COL = resolve_col(retrieval_df, ["location", "legal_location", "section"])
URL_COL = resolve_col(retrieval_df, ["url", "source_url"])

print({
    "QID_COL": QID_COL,
    "QUESTION_COL": QUESTION_COL,
    "RANK_COL": RANK_COL,
    "CHUNK_ID_COL": CHUNK_ID_COL,
    "TEXT_COL": TEXT_COL,
    "SOURCE_COL": SOURCE_COL,
    "LOCATION_COL": LOCATION_COL,
    "URL_COL": URL_COL,
})

assert QID_COL is not None
assert CHUNK_ID_COL is not None

{'QID_COL': 'qid', 'QUESTION_COL': 'question', 'RANK_COL': 'rank', 'CHUNK_ID_COL': 'chunk_id', 'TEXT_COL': 'text', 'SOURCE_COL': 'file', 'LOCATION_COL': 'location', 'URL_COL': None}


In [7]:
if isinstance(chunks, dict):
    chunk_list = chunks.get("chunks", list(chunks.values()))
else:
    chunk_list = chunks

chunk_lookup = {}
for ch in chunk_list:
    if not isinstance(ch, dict):
        continue
    cid = ch.get("chunk_id") or ch.get("id")
    if cid is not None:
        chunk_lookup[str(cid)] = ch

print("Chunk lookup:", len(chunk_lookup))

Chunk lookup: 1383


In [8]:
assert len(questions) == 10, f"Expected 10 questions, got {len(questions)}"
assert len(retrieval_df) == 50, f"Expected 50 retrieval rows, got {len(retrieval_df)}"

counts = retrieval_df.groupby(QID_COL).size().to_dict()
for qid in qid_to_question:
    assert counts.get(qid, 0) == 5, f"{qid}: expected 5 rows, got {counts.get(qid, 0)}"

if QUESTION_COL is not None:
    for qid, g in retrieval_df.groupby(QID_COL):
        vals = set(g[QUESTION_COL].dropna().astype(str))
        assert vals == {qid_to_question[qid]}, f"{qid}: retrieval question mismatch"

print("✓ Frozen Top-5 retrieval rows verified")

✓ Frozen Top-5 retrieval rows verified


In [9]:
SYSTEM_PROMPT = """
You are evaluating an EU food-regulation question.

Answer conservatively and distinguish:
1. what is supported,
2. what remains uncertain or requires verification.

Do not invent legal provisions, numerical limits, or documentary requirements.

Use this structure:
## 1. Short answer
## 2. Legal basis and relevant conditions
## 3. Uncertainties / verification needed
""".strip()

print(SYSTEM_PROMPT)

You are evaluating an EU food-regulation question.

Answer conservatively and distinguish:
1. what is supported,
2. what remains uncertain or requires verification.

Do not invent legal provisions, numerical limits, or documentary requirements.

Use this structure:
## 1. Short answer
## 2. Legal basis and relevant conditions
## 3. Uncertainties / verification needed


In [10]:
def build_prompt_only_input(question: str) -> str:
    return f"""{BENCHMARK_CONTEXT}

CURRENT QUESTION:
{question}
""".strip()


def get_retrieved_rows(qid: str) -> pd.DataFrame:
    g = retrieval_df[retrieval_df[QID_COL] == qid].copy()
    if RANK_COL is not None:
        g = g.sort_values(RANK_COL)
    return g


def row_to_evidence(row, evidence_index: int) -> str:
    cid = str(row[CHUNK_ID_COL])
    fallback = chunk_lookup.get(cid, {})

    def pick(row_col, fallback_keys):
        if row_col is not None:
            val = row.get(row_col)
            if pd.notna(val):
                return str(val)
        for k in fallback_keys:
            val = fallback.get(k)
            if val:
                return str(val)
        return ""

    text = pick(TEXT_COL, ["text", "chunk_text", "content"])
    source = pick(SOURCE_COL, ["source_file", "filename", "source"])
    location = pick(LOCATION_COL, ["location", "legal_location", "section"])
    url = pick(URL_COL, ["url", "source_url"])

    header = " | ".join([x for x in [source, location, url] if x])
    return f"[E{evidence_index}] {header}\n{text}".strip()


def build_rag_input(qid: str, question: str) -> str:
    rows = get_retrieved_rows(qid)
    evidence = "\n\n".join(
        row_to_evidence(row, i)
        for i, (_, row) in enumerate(rows.iterrows(), start=1)
    )

    return f"""{BENCHMARK_CONTEXT}

CURRENT QUESTION:
{question}

FROZEN RETRIEVED EVIDENCE:
{evidence}

Answer only the current question using the supplied evidence.
If the evidence does not support a proposition, say so rather than filling the gap from memory.
When relying on retrieved evidence, cite it using markers like [E1], [E2].
""".strip()

In [11]:
for qid in ["Q1", "Q5", "Q8", "Q9", "Q10"]:
    q = qid_to_question[qid]
    print("=" * 100)
    print(qid, "PROMPT-ONLY")
    print(build_prompt_only_input(q)[:1500])
    print()
    print(qid, "RAG")
    print(build_rag_input(qid, q)[:2400])
    print()

Q1 PROMPT-ONLY
All questions belong to one fixed EU food-regulation benchmark concerning the same case:
a shipment of dried figs originating from Türkiye in which aflatoxin B1 has been detected.

Q1–Q10 form one regulatory-assessment sequence. Answer only the current question, but when
the wording refers to “if so”, “the assessment”, “major conclusions”, “the evidence above”,
or similar cross-question references, interpret those references within this shared benchmark case.

Across the sequence, the regulatory issues being examined include:
relevant EU legislation, contaminant limits, product-use/processing distinctions,
import controls and documentation, responsible actors, possible regulatory actions,
and which conclusions require current-law verification.

Do not assume shipment-specific facts that are not stated in the current question or supplied evidence.
Do not invent legal provisions, numerical limits, or documentary requirements.

CURRENT QUESTION:
A shipment of dried figs ori

## Generation configuration

The historical context-repaired run used this sequence:

1. Inspect the input guards and repaired prompts with `RUN_GENERATION = False`.
2. Enable `RUN_GENERATION = True` for the authorized generation run.
3. Leave `SELECT_QIDS = None` to include the full Q1–Q10 benchmark.
4. Execute the generation cells for the official comparison.

The NB04b filenames were distinct from the original NB04 run. They now identify frozen records under `results/build01_02/`, not new empty outputs.

For portfolio review, keep generation disabled and inspect the saved results. Do not use Run All as a read-only check: configuration and export cells can write to existing paths even when generation is disabled. The repository's `tests/offline_smoke.py` validates selected loading and initialization cells without modifying frozen records.


In [22]:
MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"
MAX_OUTPUT_TOKENS = 4096

RUN_GENERATION = False
SELECT_QIDS = None   # None = Q1–Q10 full rerun

EXPERIMENT_ID = "nb04b_context_repaired_full_rerun_v1"

OUTPUT_JSONL = __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_generation_results.jsonl")
OUTPUT_COMPARISON_CSV = __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_answers_comparison.csv")
OUTPUT_MANUAL_EVAL_CSV = __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_manual_evaluation.csv")
OUTPUT_FAILURE_LOG_CSV = __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_failure_log.csv")
OUTPUT_CONFIG_JSON = __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_experiment_config.json")

print("RUN_GENERATION =", RUN_GENERATION)
print("SELECT_QIDS =", SELECT_QIDS)

RUN_GENERATION = True
SELECT_QIDS = None


In [23]:
experiment_config = {
    "experiment_id": EXPERIMENT_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "reasoning_effort": REASONING_EFFORT,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "benchmark_context_sha256": context_sha256,
    "benchmark_context": BENCHMARK_CONTEXT,
    "system_prompt": SYSTEM_PROMPT,
    "retrieval_config_file": str(RETRIEVAL_CONFIG_PATH),
    "retrieval_results_file": str(RETRIEVAL_CSV_PATH),
    "retrieval_frozen": True,
    "retrieval_rerun_in_nb04b": False,
    "same_context_for_prompt_only_and_rag": True,
    "previous_model_answers_in_context": False,
    "gold_reference_in_generation_prompt": False,
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(experiment_config, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_CONFIG_JSON)

Saved: data\nb04b_context_repaired_experiment_config.json


In [25]:
if RUN_GENERATION:
    from getpass import getpass
    from openai import OpenAI

    OPENAI_API_KEY = getpass("OpenAI API key: ").strip()

    if not OPENAI_API_KEY:
        raise ValueError("No API key was entered.")

    client = OpenAI(api_key=OPENAI_API_KEY)
    print("✓ OpenAI client initialized")
else:
    client = None
    print("Generation disabled — no API calls will be made.")

✓ OpenAI client initialized


In [26]:
def extract_response_text(response):
    if getattr(response, "output_text", None):
        return response.output_text

    texts = []
    for item in getattr(response, "output", []) or []:
        for c in getattr(item, "content", []) or []:
            txt = getattr(c, "text", None)
            if txt:
                texts.append(txt)
    return "\n".join(texts).strip()


def call_model(user_input: str):
    response = client.responses.create(
        model=MODEL_NAME,
        reasoning={"effort": REASONING_EFFORT},
        instructions=SYSTEM_PROMPT,
        input=user_input,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        store=False,
    )

    usage = getattr(response, "usage", None)

    return {
        "answer": extract_response_text(response),
        "response_id": getattr(response, "id", None),
        "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
        "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
        "total_tokens": getattr(usage, "total_tokens", None) if usage else None,
    }

In [27]:
def load_jsonl(path: Path):
    rows = []
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    return rows

existing = load_jsonl(OUTPUT_JSONL)
done_keys = {
    (r["qid"], r["mode"], r["model"], r["reasoning_effort"], r.get("experiment_id"))
    for r in existing
}

print("Existing repaired-run rows:", len(existing))

Existing repaired-run rows: 0


In [28]:
selected_questions = questions
if SELECT_QIDS is not None:
    wanted = set(SELECT_QIDS)
    selected_questions = [x for x in questions if x["qid"] in wanted]

print("Selected QIDs:", [x["qid"] for x in selected_questions])

Selected QIDs: ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10']


In [29]:
if RUN_GENERATION:
    with open(OUTPUT_JSONL, "a", encoding="utf-8") as fout:
        for item in selected_questions:
            qid = item["qid"]
            question = item["question"]

            for mode in ["prompt_only", "rag"]:
                key = (qid, mode, MODEL_NAME, REASONING_EFFORT, EXPERIMENT_ID)

                if key in done_keys:
                    print("Skip existing:", qid, mode)
                    continue

                if mode == "prompt_only":
                    user_input = build_prompt_only_input(question)
                    retrieved_chunk_ids = []
                else:
                    user_input = build_rag_input(qid, question)
                    retrieved_chunk_ids = (
                        get_retrieved_rows(qid)[CHUNK_ID_COL]
                        .astype(str)
                        .tolist()
                    )

                print(f"Running {qid} | {mode} ...")
                result = call_model(user_input)

                record = {
                    "experiment_id": EXPERIMENT_ID,
                    "qid": qid,
                    "question": question,
                    "mode": mode,
                    "model": MODEL_NAME,
                    "reasoning_effort": REASONING_EFFORT,
                    "benchmark_context_sha256": context_sha256,
                    "answer": result["answer"],
                    "response_id": result["response_id"],
                    "input_tokens": result["input_tokens"],
                    "output_tokens": result["output_tokens"],
                    "total_tokens": result["total_tokens"],
                    "retrieved_chunk_ids": retrieved_chunk_ids,
                    "generated_utc": datetime.now(timezone.utc).isoformat(),
                }

                fout.write(json.dumps(record, ensure_ascii=False) + "\n")
                fout.flush()
                done_keys.add(key)

    print("✓ Generation complete")
else:
    print("RUN_GENERATION=False — skipped API calls")

Running Q1 | prompt_only ...
Running Q1 | rag ...
Running Q2 | prompt_only ...
Running Q2 | rag ...
Running Q3 | prompt_only ...
Running Q3 | rag ...
Running Q4 | prompt_only ...
Running Q4 | rag ...
Running Q5 | prompt_only ...
Running Q5 | rag ...
Running Q6 | prompt_only ...
Running Q6 | rag ...
Running Q7 | prompt_only ...
Running Q7 | rag ...
Running Q8 | prompt_only ...
Running Q8 | rag ...
Running Q9 | prompt_only ...
Running Q9 | rag ...
Running Q10 | prompt_only ...
Running Q10 | rag ...
✓ Generation complete


In [30]:
results = load_jsonl(OUTPUT_JSONL)
results_df = pd.DataFrame(results)

if not results_df.empty:
    official = results_df[results_df["experiment_id"] == EXPERIMENT_ID].copy()

    comparison = (
        official.pivot_table(
            index=["qid", "question"],
            columns="mode",
            values="answer",
            aggfunc="last"
        )
        .reset_index()
        .rename(columns={
            "prompt_only": "prompt_only_answer",
            "rag": "rag_answer"
        })
    )

    q_order = {f"Q{i}": i for i in range(1, 11)}
    comparison["_order"] = comparison["qid"].map(q_order)
    comparison = comparison.sort_values("_order").drop(columns="_order")

    comparison.to_csv(OUTPUT_COMPARISON_CSV, index=False)
    display(comparison)
    print("Saved:", OUTPUT_COMPARISON_CSV)
else:
    print("No repaired-run results yet.")

mode,qid,question,prompt_only_answer,rag_answer
0,Q1,A shipment of dried figs originating from Türk...,## 1. Short answer\n\nThe main EU legislation ...,## 1. Short answer\nThe supplied evidence iden...
2,Q2,What is the applicable EU maximum level for af...,## 1. Short answer\n\nFor dried figs intended ...,## 1. Short answer\nFor dried figs intended fo...
3,Q3,Does the applicable limit change depending on ...,## 1. Short answer\n\nYes. EU maximum levels f...,## 1. Short answer\n\nThe supplied evidence do...
4,Q4,Are dried figs from Türkiye subject to increas...,## 1. Short answer\n\n**Yes.** Dried figs orig...,## 1. Short answer\n\nYes. The supplied eviden...
5,Q5,"If so, what controls, sampling/testing require...",## 1. Short answer\n\n**If the consignment’s p...,## 1. Short answer\n\nThe supplied evidence su...
6,Q6,Which authorities or food business operators a...,## 1. Short answer\n\nResponsibility is shared...,## 1. Short answer\n\nFor food that is non-com...
7,Q7,What regulatory actions may follow if aflatoxi...,## 1. Short answer\n\nIf the shipment is confi...,## 1. Short answer\n\nThe supplied evidence do...
8,Q8,Which parts of the assessment depend on curren...,## 1. Short answer\n\nThe assessment’s **legal...,## 1. Short answer\n\nThe parts of the assessm...
9,Q9,Provide the legal basis for each major conclus...,## 1. Short answer\n\nThe principal legal basi...,## 1. Short answer\n\nThe supplied evidence su...
1,Q10,"Based on the evidence above, prepare a concise...",## 1. Short answer\n\nAflatoxin B1 has been de...,## 1. Short answer\n\nThe evidence confirms th...


Saved: data\nb04b_context_repaired_openai_luna_answers_comparison.csv


In [31]:
if "comparison" in globals() and not comparison.empty:
    eval_rows = []
    for _, row in comparison.iterrows():
        for answer_col, mode in [
            ("prompt_only_answer", "prompt_only"),
            ("rag_answer", "rag"),
        ]:
            if answer_col not in comparison.columns or pd.isna(row.get(answer_col)):
                continue

            eval_rows.append({
                "qid": row["qid"],
                "mode": mode,
                "correct": "",
                "grounded": "",
                "complete": "",
                "failure_type": "",
                "failure_note": "",
            })

    manual_eval_df = pd.DataFrame(eval_rows)
    manual_eval_df.to_csv(OUTPUT_MANUAL_EVAL_CSV, index=False)
    display(manual_eval_df)
    print("Saved:", OUTPUT_MANUAL_EVAL_CSV)
else:
    print("Run generation first.")

,qid,mode,correct,grounded,complete,failure_type,failure_note
0,Q1,prompt_only,,,,,
1,Q1,rag,,,,,
2,Q2,prompt_only,,,,,
3,Q2,rag,,,,,
4,Q3,prompt_only,,,,,
5,Q3,rag,,,,,
6,Q4,prompt_only,,,,,
7,Q4,rag,,,,,
8,Q5,prompt_only,,,,,
9,Q5,rag,,,,,


Saved: data\nb04b_context_repaired_openai_luna_manual_evaluation.csv


In [32]:
failure_columns = [
    "qid",
    "mode",
    "failure_type",
    "retrieval_evidence_present",
    "notes",
]

if not OUTPUT_FAILURE_LOG_CSV.exists():
    pd.DataFrame(columns=failure_columns).to_csv(OUTPUT_FAILURE_LOG_CSV, index=False)

print("Failure log:", OUTPUT_FAILURE_LOG_CSV)

Failure log: data\nb04b_context_repaired_openai_luna_failure_log.csv


## Failure classification after rerun

Apply the frozen Notion checklist.

For Build02 RAG:

1. Required evidence absent from frozen library  
   → **source coverage failure**

2. Evidence exists in library but not frozen Top-5  
   → **retrieval failure**

3. Evidence is in frozen Top-5 but answer remains wrong/incomplete  
   → **generation/reasoning failure**

4. Question requires law newer than the frozen snapshot  
   → **currentness / static-snapshot limitation**

`BENCHMARK_CONTEXT` is not evidence and must never be counted as grounding.

## Official experiment statement

After this notebook is completed successfully:

**Same fixed benchmark context + same Q1–Q10 + same model + same system prompt + frozen Build02 Top-5 retrieval**

The architecture difference is only:

- **Build01 Prompt-only:** no retrieved evidence
- **Build02 RAG:** frozen retrieved evidence supplied

That is the cleanest comparison for this benchmark.